# LEGO Minifigure Category Predictor — Interactive Demo

Select any minifigure image from the dataset and see the model's prediction with top-5 probabilities.

In [ ]:
import json, os, random
import torch
import torch.nn as nn
from torchvision import transforms, models
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output

# ============================================================
# SETUP
# ============================================================
BASE_DIR = '/Users/linh/Downloads/KUL /11. Advanced Analytics/big_data_assignment_2'
MODEL_PATH = os.path.join(BASE_DIR, 'optionB_improved_results', 'best_model.pth')
IMG_DIR = os.path.join(BASE_DIR, 'images')
IMG_SIZE = 224

DEVICE = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')

# Same merge map as training
MERGE_MAP = {
    'The LEGO NINJAGO Movie': 'NINJAGO',
    'The LEGO Movie': 'LEGO Movies', 'The LEGO Movie 2': 'LEGO Movies',
    'DC Super Hero Girls': 'Super Heroes', 'Spider-Man': 'Super Heroes', 'Batman I': 'Super Heroes',
    'Elves': 'Friends & Fantasy', 'Friends': 'Friends & Fantasy',
    'DUPLO': 'Preschool', 'Primo': 'Preschool', 'Belville': 'Preschool', 'Scala': 'Preschool',
    'For Juniors': 'Preschool', 'Homemaker': 'Preschool', 'Fabuland': 'Preschool',
    'Pirates of the Caribbean': 'Pirates',
    'Castle': 'Castle & Medieval', 'Vikings': 'Castle & Medieval', 'Ninja': 'Castle & Medieval',
    'NEXO KNIGHTS': 'Castle & Medieval',
    'Space': 'Space & Sci-Fi', 'Aquazone': 'Space & Sci-Fi', 'Alpha Team': 'Space & Sci-Fi',
    'Exo-Force': 'Space & Sci-Fi', 'Power Miners': 'Space & Sci-Fi', 'Rock Raiders': 'Space & Sci-Fi',
    'Agents': 'Space & Sci-Fi', 'Ultra Agents': 'Space & Sci-Fi', 'Atlantis': 'Space & Sci-Fi',
    'Adventurers': 'Adventure', 'Indiana Jones': 'Adventure', "Pharaoh's Quest": 'Adventure',
    'Dino Attack': 'Adventure', 'Dino': 'Adventure', 'Western': 'Adventure',
    'SPEED CHAMPIONS': 'Racing & Vehicles', 'Racers': 'Racing & Vehicles',
    'World Racers': 'Racing & Vehicles', 'Speed Racer': 'Racing & Vehicles', 'Cars': 'Racing & Vehicles',
    'LEGENDS OF CHIMA': 'Fantasy', 'DREAMZzz': 'Fantasy', 'Hidden Side': 'Fantasy',
    'Monster Fighters': 'Fantasy',
    'The Hobbit and The Lord of the Rings': 'Middle-Earth',
    'Teenage Mutant Ninja Turtles': 'Licensed Entertainment',
    'SpongeBob SquarePants': 'Licensed Entertainment', 'Scooby-Doo': 'Licensed Entertainment',
    'The Simpsons': 'Licensed Entertainment', 'Ghostbusters': 'Licensed Entertainment',
    'Stranger Things': 'Licensed Entertainment', 'The Angry Birds Movie': 'Licensed Entertainment',
    'Trolls World Tour': 'Licensed Entertainment', 'The Incredibles': 'Licensed Entertainment',
    'Toy Story': 'Licensed Entertainment', 'Wednesday': 'Licensed Entertainment',
    'Wicked': 'Licensed Entertainment', 'Despicable Me and Minions': 'Licensed Entertainment',
    'The Lone Ranger': 'Licensed Entertainment', 'Prince of Persia': 'Licensed Entertainment',
    'Island Xtreme Stunts': 'Licensed Entertainment', 'The Powerpuff Girls': 'Licensed Entertainment',
    'Back to the Future': 'Licensed Entertainment', 'Friends TV Series': 'Licensed Entertainment',
    'Queer Eye': 'Licensed Entertainment', 'Lightyear': 'Licensed Entertainment',
    'Bluey': 'Licensed Entertainment', "Gabby's Dollhouse": 'Licensed Entertainment',
    'Dune': 'Licensed Entertainment', 'Star Trek': 'Licensed Entertainment',
    'Overwatch': 'Video Games', 'Fortnite': 'Video Games', 'Sonic the Hedgehog': 'Video Games',
    'Horizon': 'Video Games', 'The Legend of Zelda': 'Video Games', 'Animal Crossing': 'Video Games',
    'Avatar The Last Airbender': 'Video Games', 'Avatar': 'Video Games', 'One Piece': 'Video Games',
    'Collectible Minifigures': 'Collectible & Special', 'Holiday & Event': 'Collectible & Special',
    'Dimensions': 'Collectible & Special', 'Vidiyo': 'Collectible & Special',
    'Games': 'Collectible & Special',
    'LEGO Ideas': 'LEGO Promotional', 'BrickLink Designer Program': 'LEGO Promotional',
    'LEGO Brand': 'LEGO Promotional', 'LEGOLAND': 'LEGO Promotional',
    'LEGOLAND Parks': 'LEGO Promotional', 'Promotional': 'LEGO Promotional',
    'Studios': 'LEGO Promotional', 'FIRST LEGO League': 'LEGO Promotional',
    'Educational & Dacta': 'Education & Technical', 'BIONICLE': 'Education & Technical',
    'Hero Factory': 'Education & Technical', 'Technic': 'Education & Technical',
    'Basic': 'Education & Technical', 'FreeStyle': 'Education & Technical',
    'Master Builder Academy': 'Education & Technical', 'Building Bigger Thinking': 'Education & Technical',
    'Discovery': 'Education & Technical', 'Quatro': 'Education & Technical',
    'Universe': 'Education & Technical', 'Fusion': 'Education & Technical',
    'Nike': 'Education & Technical', 'Architecture': 'Education & Technical',
    'Clikits': 'Education & Technical', 'Unikitty!': 'Education & Technical',
}

print('Loading data...')

In [ ]:
# Load minifig metadata
with open(os.path.join(BASE_DIR, 'minifigs.json')) as f:
    data = json.load(f)

# Build lookup: filename -> minifig info
minifig_lookup = {}
for d in data:
    if d.get('img_local_path'):
        fname = os.path.basename(d['img_local_path'])
        d['merged_category'] = MERGE_MAP.get(d['category'], d['category'])
        minifig_lookup[fname] = d

# Build label encoding (must match training)
all_merged = sorted(set(d['merged_category'] for d in minifig_lookup.values()))
cat2idx = {c: i for i, c in enumerate(all_merged)}
idx2cat = {i: c for c, i in cat2idx.items()}
num_classes = len(all_merged)

# Get list of available images
all_images = sorted([f for f in os.listdir(IMG_DIR) if f.lower().endswith(('.jpg', '.png', '.jpeg'))])
print(f'Loaded {len(all_images)} images, {num_classes} classes')

In [ ]:
# Load model
model = models.efficientnet_b0(weights=None)
model.classifier = nn.Sequential(
    nn.Dropout(0.4),
    nn.Linear(model.classifier[1].in_features, num_classes)
)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE, weights_only=True))
model = model.to(DEVICE)
model.eval()

# Transform (same as evaluation)
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

print('Model loaded successfully!')

In [ ]:
def predict_and_show(image_filename):
    """Predict category for a single image and display results."""
    img_path = os.path.join(IMG_DIR, image_filename)
    if not os.path.exists(img_path):
        print(f'Image not found: {img_path}')
        return

    # Load and predict
    img = Image.open(img_path).convert('RGB')
    img_tensor = transform(img).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        output = model(img_tensor)
        probs = torch.softmax(output, dim=1).cpu().numpy()[0]

    # Top 5 predictions
    top5_idx = probs.argsort()[-5:][::-1]
    top5_cats = [idx2cat[i] for i in top5_idx]
    top5_probs = [probs[i] for i in top5_idx]
    predicted = top5_cats[0]

    # Ground truth
    info = minifig_lookup.get(image_filename, {})
    true_cat = info.get('merged_category', 'Unknown')
    original_cat = info.get('category', 'Unknown')
    name = info.get('name', 'Unknown')
    minifig_num = info.get('minifig_number', '?')
    year = info.get('year', '?')
    correct = predicted == true_cat

    # Plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6),
                                    gridspec_kw={'width_ratios': [1, 1.5]})

    # Left: image
    ax1.imshow(img)
    ax1.set_title(f'{minifig_num} ({year})', fontsize=12, fontweight='bold')
    ax1.axis('off')

    # Right: top-5 bar chart
    colors = ['#27ae60' if c == true_cat else '#3498db' for c in top5_cats]
    if not correct:
        colors[0] = '#e74c3c'  # Red if top prediction is wrong

    bars = ax2.barh(range(4, -1, -1), top5_probs, color=colors, edgecolor='white', linewidth=0.5)
    ax2.set_yticks(range(4, -1, -1))
    ax2.set_yticklabels([f'{c}' for c in top5_cats], fontsize=11)
    ax2.set_xlim(0, 1.0)
    ax2.set_xlabel('Probability', fontsize=11)

    # Add probability labels
    for i, (bar, prob) in enumerate(zip(bars, top5_probs)):
        ax2.text(prob + 0.01, 4 - i, f'{prob:.1%}', va='center', fontsize=10)

    result_icon = 'CORRECT' if correct else 'WRONG'
    result_color = '#27ae60' if correct else '#e74c3c'
    ax2.set_title(f'Prediction: {predicted}  [{result_icon}]\n'
                  f'True: {true_cat} (original: {original_cat})',
                  fontsize=12, fontweight='bold', color=result_color)

    # Legend
    ax2.text(0.98, -0.08, 'Green = correct class | Blue = other | Red = wrong prediction',
             transform=ax2.transAxes, fontsize=8, ha='right', color='gray')

    plt.suptitle(f'{name}', fontsize=11, y=0.02, color='gray')
    plt.tight_layout()
    plt.show()

---
## Option 1: Dropdown Selector

In [ ]:
# Interactive dropdown to pick any image
output_area = widgets.Output()

dropdown = widgets.Combobox(
    placeholder='Type minifig number (e.g. SW0001) or scroll...',
    options=all_images,
    description='Image:',
    layout=widgets.Layout(width='500px'),
    ensure_option=True
)

predict_btn = widgets.Button(
    description='Predict',
    button_style='success',
    icon='check',
    layout=widgets.Layout(width='120px')
)

def on_predict(b):
    with output_area:
        clear_output(wait=True)
        if dropdown.value and dropdown.value in all_images:
            predict_and_show(dropdown.value)
        else:
            print('Please select a valid image from the dropdown.')

predict_btn.on_click(on_predict)
display(widgets.HBox([dropdown, predict_btn]))
display(output_area)

---
## Option 2: Random Sample Button

In [ ]:
# Random sample button
random_output = widgets.Output()

random_btn = widgets.Button(
    description='Random Minifig',
    button_style='info',
    icon='random',
    layout=widgets.Layout(width='180px', height='40px')
)

def on_random(b):
    with random_output:
        clear_output(wait=True)
        img_file = random.choice(all_images)
        print(f'Selected: {img_file}')
        predict_and_show(img_file)

random_btn.on_click(on_random)
display(random_btn)
display(random_output)

---
## Option 3: Filter by Category & Browse

In [ ]:
# Filter by merged category
cat_output = widgets.Output()

cat_dropdown = widgets.Dropdown(
    options=['All'] + sorted(all_merged),
    value='All',
    description='Category:',
    layout=widgets.Layout(width='350px')
)

cat_btn = widgets.Button(
    description='Show Random from Category',
    button_style='warning',
    icon='filter',
    layout=widgets.Layout(width='250px')
)

def on_cat_filter(b):
    with cat_output:
        clear_output(wait=True)
        selected_cat = cat_dropdown.value
        if selected_cat == 'All':
            candidates = all_images
        else:
            candidates = [f for f in all_images
                         if minifig_lookup.get(f, {}).get('merged_category') == selected_cat]
        if not candidates:
            print(f'No images found for category: {selected_cat}')
            return
        img_file = random.choice(candidates)
        print(f'Category: {selected_cat} ({len(candidates)} images) | Selected: {img_file}')
        predict_and_show(img_file)

cat_btn.on_click(on_cat_filter)
display(widgets.HBox([cat_dropdown, cat_btn]))
display(cat_output)

---
## Option 4: Batch Prediction (5 Random Samples)

In [ ]:
batch_output = widgets.Output()

batch_btn = widgets.Button(
    description='Predict 5 Random',
    button_style='primary',
    icon='th',
    layout=widgets.Layout(width='180px', height='40px')
)

def on_batch(b):
    with batch_output:
        clear_output(wait=True)
        samples = random.sample(all_images, 5)
        correct_count = 0

        for img_file in samples:
            img_path = os.path.join(IMG_DIR, img_file)
            img = Image.open(img_path).convert('RGB')
            img_tensor = transform(img).unsqueeze(0).to(DEVICE)

            with torch.no_grad():
                output = model(img_tensor)
                probs = torch.softmax(output, dim=1).cpu().numpy()[0]

            pred_idx = probs.argmax()
            predicted = idx2cat[pred_idx]
            confidence = probs[pred_idx]
            info = minifig_lookup.get(img_file, {})
            true_cat = info.get('merged_category', '?')
            correct = predicted == true_cat
            if correct:
                correct_count += 1

        print(f'Batch accuracy: {correct_count}/5 ({correct_count/5*100:.0f}%)\n')

        # Show all 5
        fig, axes = plt.subplots(1, 5, figsize=(20, 5))
        for ax, img_file in zip(axes, samples):
            img_path = os.path.join(IMG_DIR, img_file)
            img = Image.open(img_path).convert('RGB')
            img_tensor = transform(img).unsqueeze(0).to(DEVICE)

            with torch.no_grad():
                output = model(img_tensor)
                probs = torch.softmax(output, dim=1).cpu().numpy()[0]

            pred_idx = probs.argmax()
            predicted = idx2cat[pred_idx]
            confidence = probs[pred_idx]
            info = minifig_lookup.get(img_file, {})
            true_cat = info.get('merged_category', '?')
            correct = predicted == true_cat

            ax.imshow(img)
            color = '#27ae60' if correct else '#e74c3c'
            icon = 'O' if correct else 'X'
            ax.set_title(f'[{icon}] {predicted}\n({confidence:.0%})\nTrue: {true_cat}',
                        fontsize=8, color=color, fontweight='bold')
            ax.axis('off')

        plt.tight_layout()
        plt.show()

batch_btn.on_click(on_batch)
display(batch_btn)
display(batch_output)